In [36]:
import pandas as pd
import japanize_matplotlib
from matplotlib import pyplot as plt
import matplotlib.cm as cm
from matplotlib.colors import to_rgba
import numpy as np
import glob
import os
import re
from enum import Enum

class Area(Enum):
    YOKOSUKA = "Yokosuka"

class ModelType(Enum):
    AGENTS_MODEL = "AgentsModel"
    
class Mode(Enum):
    TRAIN = "Train"

In [37]:
# 各種CSVデータの読み込み部
# 条件指定 TODO : enumで簡単に指定できるようにする

DATAFOLDER = "../../Master-Simulator/Assets/Data_2025-01-07_18-48-04"
SIMULATE_ID = "e3b23646-974a-44c2-bb0d-53dbfd33e94f"
AREA = Area.YOKOSUKA
MODELTYPE = ModelType.AGENTS_MODEL
MODE = Mode.TRAIN

# データ読み込み
AGENT_LOG_FOLDER = "AgentActionLogs"
ENV_EVACUATERATE_FOLDER = "EnvEvacuationRate"
SHELTER_COUNTLOG_FOLDER = "TowerEvacueeCount"
SUMMURY_FILENAME = "EpisodeSummary"

agent_action_Log_df: pd.DataFrame
env_evacuaterate_df: pd.DataFrame
shelter_countlog_df: pd.DataFrame

# エージェントの行動ログの読み込み
folder_path = f"{DATAFOLDER}/{AREA.value}_{MODELTYPE.value}_{MODE.value}_{SIMULATE_ID}/{AGENT_LOG_FOLDER}"
csv_files = glob.glob(os.path.join(folder_path, '**', '*.csv'), recursive=True)
dataframes = []
for file_path in csv_files:
    file_name = os.path.basename(file_path)
    parent_folder = os.path.basename(os.path.dirname(file_path))
    match = re.search(r'Ep-(\d+)', file_name)
    episode_num = int(match.group(1)) if match else None
    df = pd.read_csv(file_path)
    df['AgentID'] = parent_folder
    df["Episode"] = episode_num
    dataframes.append(df)

agent_action_Log_df = pd.concat(dataframes, ignore_index=True)

# 環境の避難率の読み込み
folder_path = f"{DATAFOLDER}/{AREA.value}_{MODELTYPE.value}_{MODE.value}_{SIMULATE_ID}/{ENV_EVACUATERATE_FOLDER}"
csv_files = glob.glob(os.path.join(folder_path, '*.csv'), recursive=True)
dataframes = []
for file_path in csv_files:
    file_name = os.path.basename(file_path)
    df = pd.read_csv(file_path)
    # Episode番号を取得
    match = re.search(r'Ep-(\d+)', file_name)
    episode_num = int(match.group(1)) if match else None
    df["Episode"] = episode_num
    dataframes.append(df)

env_evacuaterate_df = pd.concat(dataframes, ignore_index=True)

# 避難所毎の避難者数推移の読み込み
folder_path = f"{DATAFOLDER}/{AREA.value}_{MODELTYPE.value}_{MODE.value}_{SIMULATE_ID}/{SHELTER_COUNTLOG_FOLDER}"
csv_files = glob.glob(os.path.join(folder_path, '**', '*.csv'), recursive=True)
dataframes = []

for file_path in csv_files:
    parent_folder = os.path.basename(os.path.dirname(file_path))
    df = pd.read_csv(file_path)
    df['ShelterID'] = parent_folder
    dataframes.append(df)

shelter_countlog_df = pd.concat(dataframes, ignore_index=True)

summary_df = pd.read_csv(f"{DATAFOLDER}/{AREA.value}_{MODELTYPE.value}_{MODE.value}_{SIMULATE_ID}/{AREA.value}_{MODELTYPE.value}_{SUMMURY_FILENAME}.csv")


### エピソード毎の概要データ
| 列名                   | 説明                                                                 |
|------------------------|----------------------------------------------------------------------|
| `index`                | エピソード番号。各エピソードの識別子として使用されます。             |
| `Limit Time`           | 制限時間（秒）。各エピソードの最大許容時間を示します。               |
| `End Time Sec`         | エピソードが終了した経過時間（秒）。エピソードの実際の終了時間です。 |
| `Total Evacuee Count`  | スポーンした避難者の総数。シミュレーション中に生成された避難者の数。 |
| `Drone Count`          | ドローンの総数。シミュレーションに参加したドローンの数。             |
| `Final Evacuate Rate`  | 最終的な避難完了率。避難者のうち、最終的に避難を完了した割合。       |


In [38]:
summary_df


,Limit Time,End Time Sec,Total Evacuee Count,Drone Count,Final Evacuate Rate
0,2045.323,2045.323,200,4,0.66
1,2054.843,2054.843,200,4,0.73
2,2184.085,1355.640,200,4,1.00


### エージェントの行動ログデータ
| 列名               | 説明                                                                 |
|--------------------|----------------------------------------------------------------------|
| `Elapsed Sec`      | 経過時間。シミュレーション開始からの時間を秒単位で示します。         |
| `Destination`      | エージェントが選択した避難所。エージェントが向かっている避難所の名称。|
| `Speed`            | エージェントが選択した移動速度。エージェントの現在の移動速度を示します。|
| `Guided Evacuees`  | エージェントが誘導中の避難者数。現在エージェントが誘導している避難者の数。|
| `AgentID`          | エージェントID。各エージェントの一意の識別子。                       |
| `Episode`          | エピソード番号。シミュレーションのエピソードを識別する番号。         |


In [39]:
agent_action_Log_df

,Elapsed Sec,Destination,Speed,Guided Evacuees,AgentID,Episode
0,0.00000,TowerC,3,55,NavAgent (1),0
1,80.33853,TowerC,3,55,NavAgent (1),0
2,80.67847,TowerC,3,55,NavAgent (1),0
3,81.01842,TowerC,3,55,NavAgent (1),0
4,81.33836,TowerC,3,55,NavAgent (1),0
...,...,...,...,...,...,...
2672,1247.83500,TowerB,3,1,NavAgent (4),2
2673,1248.17500,TowerB,3,1,NavAgent (4),2
2674,1248.49600,TowerB,3,1,NavAgent (4),2
2675,1248.83600,TowerB,3,1,NavAgent (4),2


### 環境全体の避難率推移データ
| 列名            | 説明                                                                 |
|-----------------|----------------------------------------------------------------------|
| `Elapsed Sec`   | 経過時間。シミュレーション開始からの時間を秒単位で示します。         |
| `Evacuate Rate` | 避難率。避難者のうち、避難を完了した割合を示します。                 |
| `Episode`       | エピソード番号。各シミュレーションの識別子として使用されます。       |


In [40]:
env_evacuaterate_df

,Elapsed Sec,Evacuation Rate,Episode
0,0.02,0.000,0
1,0.04,0.000,0
2,0.06,0.000,0
3,0.08,0.000,0
4,0.10,0.000,0
...,...,...,...
272614,1355.56,0.995,2
272615,1355.58,0.995,2
272616,1355.60,0.995,2
272617,1355.62,0.995,2


### 避難所毎の避難者数推移データ
| 列名            | 説明                                                                 |
|-----------------|----------------------------------------------------------------------|
| `Elapsed Sec`   | 経過時間。シミュレーション開始からの時間を秒単位で示します。         |
| `Evacuate Rate` | 避難率。避難者のうち、避難を完了した割合を示します。                 |
| `Episode`       | エピソード番号。各シミュレーションの識別子として使用されます。       |


In [41]:
shelter_countlog_df

,Elapsed Sec,Evacuee Count,ShelterID
0,0.340000,0,TowerA
1,0.679999,0,TowerA
2,1.019999,0,TowerA
3,1.339999,0,TowerA
4,1.679999,0,TowerA
...,...,...,...
65423,1354.279000,50,TowerD
65424,1354.599000,50,TowerD
65425,1354.939000,50,TowerD
65426,1355.280000,50,TowerD
